# Animation Generation

Create timeline animations and 3D movies from simulation results.

## Features:
- Timeline animations showing civilization evolution
- 3D galaxy flythrough movies
- MP4 export for presentations
- Customizable frame rate and duration

**Requirements:** 
- Simulation with snapshots enabled
- ffmpeg installed for video generation

---

## 1. Setup

In [ ]:
from great_silence.notebook import SimulationWidget, AnimationBuilder, configure_notebook_display
from great_silence.visualization import TimelineAnimator
import matplotlib.pyplot as plt
from pathlib import Path

configure_notebook_display()

# Create output directory
Path('output/animations').mkdir(parents=True, exist_ok=True)

## 2. Run Simulation with Snapshots

First, run a simulation with snapshot saving enabled.

In [ ]:
# Create simulation widget
widget = SimulationWidget()
# Display and run
widget.display();
print("\n⚠️ Make sure to run the simulation before proceeding!")
print("Snapshots are automatically enabled for animations.")

## 3. Create Timeline Animation

Generate animation showing civilization emergence and extinction over time.

In [ ]:
if widget.runner and widget.runner.simulation:
    # Create timeline animator
    animator = TimelineAnimator(widget.runner.simulation)
    
    # Generate animation
    print("Creating timeline animation...")
    anim = animator.create_animation(
        save_path='output/animations/timeline.mp4',
        interval=50,  # milliseconds between frames
        fps=20
    )
    
    print("✓ Animation saved to: output/animations/timeline.mp4")
else:
    print("Run simulation first!")

## 4. Use Animation Builder Widget

Interactive widget for customizing animation parameters.

In [ ]:
if widget.results:    # Create animation builder    anim_builder = AnimationBuilder(widget.results)        # Display interface    anim_builder.display();else:    print("Run simulation first!")

## 5. Create Static Snapshot Sequence

Generate individual frames as PNG images.

In [ ]:
if widget.runner and widget.runner.simulation:
    from great_silence.visualization import GalaxyVisualizer
    
    # Get snapshots
    snapshots = widget.runner.simulation.snapshots if hasattr(widget.runner.simulation, 'snapshots') else []
    
    if snapshots:
        print(f"Found {len(snapshots)} snapshots")
        
        # Create frame directory
        frame_dir = Path('output/animations/frames')
        frame_dir.mkdir(parents=True, exist_ok=True)
        
        viz = GalaxyVisualizer()
        
        # Generate frames (sample every 10th snapshot)
        for i, snapshot in enumerate(snapshots[::10]):
            positions = widget.runner.simulation.galaxy.positions
            
            # Get active civilizations at this time
            time_gyr = snapshot.get('time_gyr', 0)
            active_civs = [
                civ for civ in widget.runner.simulation.civilizations
                if civ.emergence_time_gyr <= time_gyr and civ.is_active
            ]
            
            civ_indices = [civ.parent_star_idx for civ in active_civs]
            
            # Create frame
            fig, ax = plt.subplots(figsize=(10, 10))
            ax.scatter(positions[:, 0], positions[:, 1], s=0.1, c='white', alpha=0.3)
            
            if civ_indices:
                civ_pos = positions[civ_indices]
                ax.scatter(civ_pos[:, 0], civ_pos[:, 1], s=50, c='red', alpha=0.8)
            
            ax.set_xlim(-20, 20)
            ax.set_ylim(-20, 20)
            ax.set_facecolor('black')
            ax.set_title(f'Time: {time_gyr:.2f} Gyr | Active Civs: {len(active_civs)}', 
                        color='white', fontsize=14)
            ax.set_xlabel('X (kpc)', color='white')
            ax.set_ylabel('Y (kpc)', color='white')
            
            # Save frame
            plt.savefig(frame_dir / f'frame_{i:04d}.png', 
                       facecolor='black', dpi=100, bbox_inches='tight')
            plt.close()
        
        print(f"✓ Frames saved to: {frame_dir}")
        print(f"\nCreate video with ffmpeg:")
        print(f"  ffmpeg -framerate 10 -pattern_type glob -i '{frame_dir}/*.png' ")
        print(f"         -c:v libx264 -pix_fmt yuv420p output/animations/custom.mp4")
    else:
        print("No snapshots available. Enable snapshots in simulation config.")
else:
    print("Run simulation first!")

## 6. Create 3D Rotating Galaxy Movie

Generate a 3D visualization with camera rotation.

In [ ]:
if widget.runner:
    import plotly.graph_objects as go
    from plotly.offline import plot
    import numpy as np
    
    positions = widget.runner.simulation.galaxy.positions
    
    # Create frames for rotation
    n_frames = 360  # One full rotation
    frames = []
    
    for angle in range(0, 360, 2):  # Every 2 degrees
        # Camera position (rotating around Z axis)
        rad = np.radians(angle)
        camera = dict(
            eye=dict(
                x=30 * np.cos(rad),
                y=30 * np.sin(rad),
                z=10
            )
        )
        
        frame = go.Frame(
            name=str(angle),
            layout=dict(scene_camera=camera)
        )
        frames.append(frame)
    
    # Create figure with animation
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=positions[:, 0],
                y=positions[:, 1],
                z=positions[:, 2],
                mode='markers',
                marker=dict(size=1, color='white', opacity=0.3)
            )
        ],
        frames=frames
    )
    
    # Add play button
    fig.update_layout(
        title='Rotating Galaxy View',
        scene=dict(
            xaxis_title='X (kpc)',
            yaxis_title='Y (kpc)',
            zaxis_title='Z (kpc)',
            bgcolor='black'
        ),
        updatemenus=[{
            'type': 'buttons',
            'showactive': False,
            'buttons': [
                {
                    'label': 'Play',
                    'method': 'animate',
                    'args': [None, {'frame': {'duration': 50}, 'fromcurrent': True}]
                },
                {
                    'label': 'Pause',
                    'method': 'animate',
                    'args': [[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]
                }
            ]
        }]
    )
    
    # Save as HTML
    html_path = 'output/animations/rotating_galaxy.html'
    fig.write_html(html_path)
    print(f"✓ Interactive 3D animation saved to: {html_path}")
    print("Open in browser to view rotating galaxy!")
    
    # Display in notebook
    fig.show()
else:
    print("Run simulation first!")

## 7. Check ffmpeg Installation

In [ ]:
import subprocess

try:
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print("✓ ffmpeg is installed:")
    print(result.stdout.split('\n')[0])
except FileNotFoundError:
    print("⚠️ ffmpeg not found!")
    print("Install with: brew install ffmpeg (macOS)")
    print("Or: conda install ffmpeg")

---

## Animation Tips

### Frame Rate Guidelines:
- **10 FPS**: Slow, good for detailed viewing
- **20 FPS**: Standard, smooth enough for most presentations
- **30 FPS**: Smooth, professional quality
- **60 FPS**: Very smooth, large file size

### Duration Recommendations:
- **30-60s**: Quick overview for talks
- **60-120s**: Detailed presentation
- **120-300s**: Full simulation playback

### File Size Optimization:
- Reduce resolution: smaller DPI in matplotlib
- Lower frame rate: fewer frames per second
- Sample snapshots: use every Nth snapshot
- Compress: use h264 codec with CRF setting

### Advanced ffmpeg Options:
```bash
# High quality
ffmpeg -framerate 30 -i frames/frame_%04d.png -c:v libx264 -crf 18 output.mp4

# Smaller file size
ffmpeg -framerate 20 -i frames/frame_%04d.png -c:v libx264 -crf 28 output.mp4

# Create GIF instead
ffmpeg -framerate 10 -i frames/frame_%04d.png -vf "fps=10,scale=800:-1" output.gif
```

---

## Next Steps

- **01_quickstart_production_workflow.ipynb**: Run new simulations
- **02_interactive_exploration.ipynb**: Analyze saved results

Happy animating! 🎬🌌